## Import libraries

In [11]:
import os
import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

## Functions

In [12]:
def train_process(train_file, model_name):
    """
    Full process: data loading, preprocessing, SMOTE, training, and saving.
    """
    print(f"\n--- Starting training: {model_name} ---")
    # 1. Load data
    df = pd.read_csv(train_file)
    y = df['target'] - 1  # Shift classes 1-7 to 0-6
    # 2. Drop columns
    static_drop = ['target', 'filename', 'edge_vector']
    cols_to_drop = [c for c in df.columns if 'z' in c or c in static_drop]
    X_raw = df.drop(columns=cols_to_drop)
    feature_names = X_raw.columns.tolist()
    # 3. Imputation (filling missing values)
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X_raw)
    # 4. Balancing (SMOTE)
    print("Applying SMOTE...")
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_imputed, y)
    # 5. Model configuration and training
    model = XGBClassifier(
        n_estimators=100,
        eval_metric='mlogloss',
        random_state=42
    )
    print("Training XGBoost model...")
    model.fit(X_resampled, y_resampled)
    # 6. Saving artifacts
    save_dir = os.path.join(os.path.join('MODELS', 'INPUT - VECTORS AND LANDMARKS'), model_name)
    os.makedirs(save_dir, exist_ok=True)
    joblib.dump(model, os.path.join(save_dir, 'model.pkl'))
    joblib.dump(imputer, os.path.join(save_dir, 'imputer.pkl'))
    joblib.dump(feature_names, os.path.join(save_dir, 'features.pkl'))
    print(f"Model and tools saved in: {save_dir}")

## Trening data without mask

In [13]:
train_process(
    train_file='DATA\VECTORS AND LANDMARKS\without_mask_train.csv',
    model_name='XGBoost_without_mask_smote'
)


--- Starting training: XGBoost_without_mask_smote ---
Applying SMOTE...
Training XGBoost model...
Model and tools saved in: MODELS\INPUT - VECTORS AND LANDMARKS\XGBoost_without_mask_smote


## Trening data with mask

In [14]:
train_process(
    train_file='DATA\VECTORS AND LANDMARKS\with_mask_train.csv',
    model_name='XGBoost_with_mask_smote'
)


--- Starting training: XGBoost_with_mask_smote ---
Applying SMOTE...
Training XGBoost model...
Model and tools saved in: MODELS\INPUT - VECTORS AND LANDMARKS\XGBoost_with_mask_smote
